In [ ]:
import os, pathlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, ConfusionMatrixDisplay
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE
print('TensorFlow:', tf.__version__)

In [ ]:
(raw_train, raw_val, raw_test), ds_info = tfds.load(
    'cats_vs_dogs', split=['train[:20%]', 'train[70%:75%]', 'train[95%:]'],
    as_supervised=True, shuffle_files=True, with_info=True)
class_names = ds_info.features['label'].names
print('Clases:', class_names)
print('Ejemplo de especificación:', raw_train.element_spec)
for name, ds in [('train', raw_train), ('validation', raw_val), ('test', raw_test)]:
    print(name, tf.data.experimental.cardinality(ds).numpy())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for ax, (img, label) in zip(axes.ravel(), raw_train.take(8)):
    ax.imshow(img.numpy())
    ax.set_title(f'{class_names[int(label)]}\n{tuple(img.shape)}')
    ax.axis('off')
plt.tight_layout(); plt.show()


sample_labels = [int(y.numpy()) for _, y in raw_train.take(500)]
print(pd.Series(sample_labels).map(dict(enumerate(class_names))).value_counts())

In [ ]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.int32)

def augment(image, label):
    image = tf.image.random_brightness(image, max_delta=0.12)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def make_dataset(raw, training=False):
    ds = raw.map(preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(raw_train, training=True)
val_ds = make_dataset(raw_val)
test_ds = make_dataset(raw_test)
for x_batch, y_batch in train_ds.take(1):
    print('x:', x_batch.shape, x_batch.dtype, '| y:', y_batch.shape, y_batch.dtype)
    print('Rango de píxeles:', float(tf.reduce_min(x_batch)), float(tf.reduce_max(x_batch)))

In [ ]:
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(images[k].numpy())
    ax.set_title(class_names[int(labels[k])]); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
BACKBONE =  'VGG16'#'ResNet50'
RUN_IMAGENET_INFERENCE = True
RUN_SCRATCH = False  
RUN_FINETUNE = True
EPOCHS_HEAD = 1
EPOCHS_FT = 1
MAX_EPOCHS_TUNER = 2
OUT_DIR = pathlib.Path('resultados_resnet_vgg_catsdogs')
OUT_DIR.mkdir(exist_ok=True, parents=True)

APPLICATIONS = {
    'ResNet50': (tf.keras.applications.ResNet50,
                 tf.keras.applications.resnet50.preprocess_input,
                 tf.keras.applications.resnet50.decode_predictions),
    'VGG16': (tf.keras.applications.VGG16,
              tf.keras.applications.vgg16.preprocess_input,
              tf.keras.applications.vgg16.decode_predictions),
}
assert BACKBONE in APPLICATIONS
print('Modelo:', BACKBONE)

In [ ]:
import glob
def imagenet_inference(name=BACKBONE,folder=False):
    constructor, preprocess_fn, decoder = APPLICATIONS[name]
    imagenet_model = constructor(weights='imagenet', include_top=True)
    if folder:
        animls = list(glob.glob('../imgs/animales/*'))
        image_raw = tf.io.read_file(animls[0])
        images = tf.image.decode_jpeg(image_raw, channels=3) # channels=3 asegura RGB
        image = tf.image.resize(images, [224, 224])
        image = tf.expand_dims(image, axis=0)
    else:
        images, _ = next(iter(test_ds))
        image = tf.image.resize(images[:1]*255., (224, 224))
    print(image.shape)
    scores = imagenet_model.predict(preprocess_fn(tf.identity(image)), verbose=0)
    plt.figure(figsize=(4, 4))
    plt.imshow(tf.clip_by_value(image[0] / 255.0, 0, 1))
    plt.axis('off'); plt.title(f'{name} — ImageNet'); plt.show()
    print('Top 3:', decoder(scores, top=1)[0])
    del imagenet_model

if RUN_IMAGENET_INFERENCE:
    imagenet_inference(folder=True)

In [ ]:
def build_transfer_model(name=BACKBONE, weights='imagenet'):
    constructor, preprocess_fn, _ = APPLICATIONS[name]

    backbone = constructor(include_top=False,
                           weights=weights,
                           input_shape=(*IMAGE_SIZE, 3))
    
    backbone.trainable = (weights is None)

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='imagen')
    x = layers.Rescaling(255.0, name='restore_pixel_scale')(inputs)# ImageNet necesita [0,255] antes de preprocess_input
    x = layers.Lambda(preprocess_fn, name='preprocess_imagenet')(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    outputs = layers.Dense(2, activation='softmax', name='probabilidades')(x)
    model = tf.keras.Model(inputs, outputs, name=name + '_Gatos_Perros')
    return model, backbone

In [ ]:
scratch_model, scratch_backbone = build_transfer_model(weights=None)
scratch_model.summary()
scratch_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

In [ ]:
scratch_history = scratch_model.fit(train_ds, validation_data=val_ds, steps_per_epoch=40,verbose=1,validation_steps=40)

In [ ]:
results = scratch_model.evaluate(test_ds, return_dict=True)
print('Métricas test:', results)
y_true = np.concatenate([yb.numpy() for _, yb in test_ds])
pred_prob = scratch_model.predict(test_ds, verbose=0)
y_pred = np.argmax(pred_prob, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print('ROC-AUC:', roc_auc_score(y_true, pred_prob[:, 1]))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — test'); plt.show()

In [ ]:
# Comparar probabilidades en unas pocas imágenes del conjunto de prueba
sample_x, sample_y = next(iter(test_ds))
sample_prob = scratch_model.predict(sample_x, verbose=0)
fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for k, ax in enumerate(axs.ravel()):
    ax.imshow(sample_x[k])
    idx = int(np.argmax(sample_prob[k]))
    ax.set_title(f'Real: {class_names[int(sample_y[k])]}\nPred: {class_names[idx]} ({sample_prob[k,idx]:.2f})')
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
TL_model, TL_backbone = build_transfer_model()
TL_model.summary()
TL_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

In [ ]:
TL_history = TL_model.fit(train_ds, validation_data=val_ds, steps_per_epoch=40*2,verbose=1,validation_steps=40)

In [ ]:
results = TL_model.evaluate(test_ds, return_dict=True)
print('Métricas test:', results)
y_true = np.concatenate([yb.numpy() for _, yb in test_ds])
pred_prob = TL_model.predict(test_ds, verbose=0)
y_pred = np.argmax(pred_prob, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print('ROC-AUC:', roc_auc_score(y_true, pred_prob[:, 1]))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — test'); plt.show()

In [ ]:
def enable_finetuning(backbone, last_n=15):
    backbone.trainable = True
    for layer in backbone.layers[:-last_n]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    print('Capas backbone entrenables:', sum(layer.trainable for layer in backbone.layers),
          '/', len(backbone.layers))

ft_model, ft_backbone = build_transfer_model(BACKBONE, weights='imagenet')
enable_finetuning(ft_backbone, last_n=2)

ft_model.summary()
ft_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
    

In [ ]:
ft_history = ft_model.fit(train_ds, validation_data=val_ds, steps_per_epoch=40,verbose=1,validation_steps=40)

In [ ]:
results = ft_model.evaluate(test_ds, return_dict=True)
print('Métricas test:', results)
y_true = np.concatenate([yb.numpy() for _, yb in test_ds])
pred_prob = ft_model.predict(test_ds, verbose=0)
y_pred = np.argmax(pred_prob, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
print('ROC-AUC:', roc_auc_score(y_true, pred_prob[:, 1]))
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', values_format='d')
plt.title('Matriz de confusión — test'); plt.show()